# Lab L2303: Classify skill archetypes and author one

```yaml
title: "Lab L2303: Classify skill archetypes and author one"
keywords: skill archetype, script-centric, principle-centric, procedure-centric, classify, author SKILL.md, discriminating description, wrong shape, get_order, refund-policy
estimated duration: 30
```

<a id="top"></a>

> **Lesson:** L23 — Skill patterns & composition. This is the lab paired with the archetypes
> lecture ([L2302_lecture.md](L2302_lecture.md)); do that reading first.
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L23/objectives.md)
> **Anchor model: Claude Sonnet 4.6.**
> **Runs offline, no key.** This whole lab is pure Python and string-writing — you classify
> real skills and author skill text by hand. There are no live model calls, so a keyless
> restart-and-run-all passes.

This lab lands **Objective 1** (classify a skill by archetype) and **Objective 2** (author each
archetype). You'll practice on the repo's own skills and the L23 example skills — not invented toys.


## Learning objectives

By the end of this lab you'll be able to:

- **Classify** a skill by its archetype — *script-centric* (API/integration recipe),
  *principle-centric* (review/rubric), or *procedure-centric* (operating-model runbook) — from a
  one-line blurb.
- **Spot a mis-shaped body** — prose that describes a deterministic step instead of scripting it —
  and name the fix.
- **Author a script-centric body** that says *when* to run a script, the *exact* invocation, and
  *how to read* the output.
- **Write a discriminating description** that stays mutually distinct from a colliding sibling skill.
- **Grow the single skill you authored in L22** — name its archetype and tighten its description to
  survive in a set.


## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — classify five skills](#2-problem-1--classify-five-skills)
- [3. Problem 2 — spot the wrong shape](#3-problem-2--spot-the-wrong-shape)
- [4. Problem 3 — author a script-centric body](#4-problem-3--author-a-script-centric-body)
- [5. Problem 4 — a discriminating description](#5-problem-4--a-discriminating-description)
- [6. Problem 5 — grow your L22 skill](#6-problem-5--grow-your-l22-skill)
- [7. Self-check](#7-self-check)


## 1. Setup

The setup cell defines the shared L23 runtime pieces this lab needs — the `Skill` dataclass and the
three archetype labels — plus a few short skill **blurbs** to classify. Everything here is offline:
no key, no network, no model call. Run it once, top to bottom.

> `Skill` and the three-archetype vocabulary are the same ones the lecture and the other L23
> notebooks use. The lab doesn't drive the agent, so we skip the `FakeModel` / `create_agent`
> machinery those notebooks carry.

[↑ Back to top](#top)


In [1]:
from __future__ import annotations

from dataclasses import dataclass

# The three archetypes from L2302, as a frozen vocabulary the asserts check against.
ARCHETYPES: tuple[str, str, str] = (
    "script-centric",  # center of gravity: a bundled script (API/integration recipe)
    "principle-centric",  # center of gravity: a rule list (review / rubric)
    "procedure-centric",  # center of gravity: an ordered procedure (operating-model runbook)
)


@dataclass(frozen=True)
class Skill:
    """A skill: a name, a description that says WHEN it applies, and a body."""

    name: str
    description: str
    body: str


# Five short skill blurbs to classify in Problem 1 — name -> one-line "what it does".
# They deliberately span the three archetypes (with two extras) so the mapping isn't 1:1 obvious.
BLURBS: dict[str, str] = {
    "get-order": (
        "Wrap a mock orders API: given an order id, run get_order_api.py and hand back its JSON."
    ),
    "pr-review": (
        "Hold a diff against a list of standards (naming, tests, types) and flag each violation."
    ),
    "release-a-version": (
        "Bump the version, update the changelog, tag the commit, and push — in that exact order."
    ),
    "format-code": ("Run the ruff formatter over the repo and report what changed."),
    "commit-style": (
        "Judge a commit message against the Conventional Commits rules and say if it conforms."
    ),
}

print("archetypes:", ARCHETYPES)
print("blurbs to classify:", list(BLURBS))

archetypes: ('script-centric', 'principle-centric', 'procedure-centric')
blurbs to classify: ['get-order', 'pr-review', 'release-a-version', 'format-code', 'commit-style']


## 2. Problem 1 — classify five skills

`BLURBS` (from setup) holds five skills, each a `name -> one-line "what it does"`. Read each blurb
and decide its **center of gravity** — is the real content a bundled **script** (script-centric), a
**rule list** you apply as judgment (principle-centric), or an ordered **procedure** you run in
sequence (procedure-centric)?

Fill `CLASSIFICATION` with one archetype string per skill. Use only the exact labels in
`ARCHETYPES`. There are examples of all three shapes in the five — read for the *shape*, not the
domain: "run this script" vs. "hold the work to these rules" vs. "do these steps in order".

[↑ Back to top](#top)


In [2]:
# One archetype per skill, chosen by its center of gravity (not its domain).
CLASSIFICATION: dict[str, str] = {
    # thin wrapper over get_order_api.py — the real work is the script
    "get-order": "script-centric",
    # a diff held against a list of standards, applied as judgment — a rubric
    "pr-review": "principle-centric",
    # bump -> changelog -> tag -> push, in that exact order — an ordered procedure
    "release-a-version": "procedure-centric",
    # a thin wrapper over the ruff formatter — the real work is the script
    "format-code": "script-centric",
    # a message judged against the Conventional Commits rules — a rubric
    "commit-style": "principle-centric",
}

# --- checks (leave these) ---
assert set(CLASSIFICATION) == set(BLURBS), "classify all five skills, by their exact names"
assert all(v in ARCHETYPES for v in CLASSIFICATION.values()), (
    "use only the exact labels in ARCHETYPES"
)
print("classification:")
for name, archetype in CLASSIFICATION.items():
    print(f"  {name}: {archetype}")

classification:
  get-order: script-centric
  pr-review: principle-centric
  release-a-version: procedure-centric
  format-code: script-centric
  commit-style: principle-centric


## 3. Problem 2 — spot the wrong shape

Below is a skill body someone wrote for a `get-order` step. It **prose-describes a deterministic
step in three verbose sentences** instead of just scripting it:

> To fetch an order, first recall that the orders API takes a single order id like `ORD-1001` and
> returns a JSON record whose `payment` object changes shape depending on the `method` field, so you
> should think carefully about which of `brand`, `email`, or `code_last4` will be present. Next,
> mentally reconstruct the pricing breakdown by remembering that `total_cents` is the sum of
> `subtotal_cents` minus `discount_cents` plus `tax_cents` plus `shipping_cents`. Finally, write out
> the fields you inferred, taking care to guess the fulfillment `tracking` block only when the order
> has shipped.

This is the **wrong-shape mistake** from L2302 §4.3. Answer the three questions in the cell below.

_Refer back to the real [`example_skills/get_order/SKILL.md`](example_skills/get_order/SKILL.md) —
that's what the right shape looks like._


**Your answer** (edit this cell):

1. **Which archetype is this body mis-written as?**

   It reads like a **principle-centric / procedure-centric** body — a paragraph of judgment and
   remembered rules ("think carefully about which fields will be present", "mentally reconstruct the
   pricing", "guess the tracking block"). It's asking the model to *reason through* a deterministic
   contract in prose.

2. **Which archetype should it be?**

   **Script-centric.** Fetching one order by id is a deterministic operation with a fixed contract —
   `get_order_api.py` already knows the shape. There is nothing to judge or re-derive.

3. **What is the fix — in one sentence?**

   Delete the prose and make the body a thin wrapper: say *when* to run the script, give the *exact*
   invocation (`uv run python -m ...get_order_api <ORDER_ID>`), and say *how to read* its JSON output
   — pushing the determinism into the script instead of the model's head.


## 4. Problem 3 — author a script-centric body

Now write the *right* shape. Author a **script-centric** `SKILL.md` body for a thin wrapper over
[`example_skills/get_order/get_order_api.py`](example_skills/get_order/get_order_api.py). A good
script-centric body (L2302 §2.1 / §4.1) is **minimal prose** and covers exactly three things:

1. **When to run it** — the trigger (you have a known order id and need its full record).
2. **The exact invocation** — the literal command, including the module path and the `<ORDER_ID>`
   argument.
3. **How to read the output** — what stdout/stderr and the exit code mean, and the one gotcha
   (read `payment.method` first; `tracking` only appears once shipped).

Set `SKILL_BODY` to that markdown string. Keep the deterministic logic in the *script*, not the
prose — you are wrapping, not re-deriving. The checks look for the script name and a sample order id.

[↑ Back to top](#top)


In [3]:
# A thin, script-centric body: when to run it, the exact call, how to read the output.
SKILL_BODY: str = """## When to run it

Run this when you have a specific **order id** and need that order's full record. If you only have a
customer name, an email, or a date range, this is the wrong skill — it does an id lookup and nothing
else.

## How to call it

From the repo root, run the bundled script with the order id as its only argument:

```sh
uv run python -m fluffy_potato_curriculum.lessons.L23.example_skills.get_order.get_order_api <ORDER_ID>
```

Known ids for the mock: `ORD-1001` (card), `ORD-1002` (paypal, shipped), `ORD-1003` (gift card,
refunded).

## How to read the output

- **Success:** indented JSON on **stdout**, exit code `0`. Read `payment.method` **first** — it tells
  you which other `payment.*` fields exist. `fulfillment.tracking` is present only once the order has
  shipped. Hand the parsed fields back as they are; don't reformat the JSON.
- **Unknown id:** a JSON error object on **stderr**, exit code `1`. Don't retry the same id.
- **Wrong usage** (no id or more than one): JSON usage error on stderr, exit code `2`.

The script is the contract; this body just says when and how to run it."""

# --- checks (leave these) ---
assert "get_order_api" in SKILL_BODY, "name the script the body wraps"
assert "ORD-" in SKILL_BODY, "show a concrete example order id so the invocation is copy-pasteable"
assert len(SKILL_BODY) > 80, "a usable body needs when + how-to-call + how-to-read, not one line"
print(SKILL_BODY)

## When to run it

Run this when you have a specific **order id** and need that order's full record. If you only have a
customer name, an email, or a date range, this is the wrong skill — it does an id lookup and nothing
else.

## How to call it

From the repo root, run the bundled script with the order id as its only argument:

```sh
uv run python -m fluffy_potato_curriculum.lessons.L23.example_skills.get_order.get_order_api <ORDER_ID>
```

Known ids for the mock: `ORD-1001` (card), `ORD-1002` (paypal, shipped), `ORD-1003` (gift card,
refunded).

## How to read the output

- **Success:** indented JSON on **stdout**, exit code `0`. Read `payment.method` **first** — it tells
  you which other `payment.*` fields exist. `fulfillment.tracking` is present only once the order has
  shipped. Hand the parsed fields back as they are; don't reformat the JSON.
- **Unknown id:** a JSON error object on **stderr**, exit code `1`. Don't retry the same id.
- **Wrong usage** (no id or more than one): J

## 5. Problem 4 — a discriminating description

The **description** is the sentence the model matches on to decide whether to read a skill at all
(L22's load-bearing craft point). Across a *set* of skills, descriptions must stay **mutually
distinct** (L2302 §4.2), or the model can't reliably pick.

Here is a colliding **sibling** skill your `get-order` skill sits next to:

```text
search-orders: Find orders matching a customer name, email, or date range, returning a list of
matching order ids and summaries. Use for discovery when you don't yet know the exact order id.
```

Write a `DESCRIPTION` for **your `get-order` skill** that is *mutually discriminating*: it should say
what it **is** for (a single-order lookup by known id) and imply what it is **not** for (searching /
listing — that's `search-orders`). The checks require a real sentence and that it differs from the
sibling's.

[↑ Back to top](#top)


In [4]:
SIBLING_DESCRIPTION = (
    "Find orders matching a customer name, email, or date range, returning a list of matching "
    "order ids and summaries. Use for discovery when you don't yet know the exact order id."
)

# A description that names the single-id lookup and rules out the sibling's search role.
DESCRIPTION: str = (
    "Fetch one order's full record from the mock orders API by its exact order id, returning the "
    "raw JSON contract (customer, line items, payment, pricing, fulfillment). Use when you already "
    "know the order id; not for searching or listing orders by name, email, or date — use "
    "search-orders for that."
)

# --- checks (leave these) ---
assert len(DESCRIPTION) > 40, "a description needs enough words to actually discriminate"
assert DESCRIPTION.strip() != SIBLING_DESCRIPTION.strip(), (
    "your description must be distinct from the sibling's"
)
print(DESCRIPTION)

Fetch one order's full record from the mock orders API by its exact order id, returning the raw JSON contract (customer, line items, payment, pricing, fulfillment). Use when you already know the order id; not for searching or listing orders by name, email, or date — use search-orders for that.


## 6. Problem 5 — grow your L22 skill

Open-ended, and the payoff of the whole lab. In [L22](../L22/L2201_intro.md) you authored **one**
`SKILL.md`. Now grow it with what L23 gave you:

1. **Name its archetype.** Which of the three is it — script-centric, principle-centric, or
   procedure-centric? Answer in the written cell below.
2. **Tighten its description to survive in a set.** Rewrite the description so it stays distinct
   when it sits next to a sibling skill in the same domain — say what it's for and imply what it's
   not. Set `GROWN_DESCRIPTION` in the code cell.

If you don't have your L22 skill handy, use the running customer-support domain from L22 and grow a
`refund-policy` skill (a rubric of when refunds are allowed), keeping it distinct from a sibling
`process-refund` skill (the runbook that actually issues one).

[↑ Back to top](#top)

**Your L22 skill's archetype** (edit this cell):

- **Skill name:** `refund-policy` (worked example in the L22 customer-support domain)
- **Archetype (script- / principle- / procedure-centric):** **principle-centric** (a review/rubric).
- **Why (one line — what's its center of gravity?):** it's a **rule list** — the conditions under
  which a refund is *allowed* — that the agent applies as judgment to a request; there's no single
  deterministic answer and no script that could replace the taste, so it's a rubric, not a runbook.


In [5]:
# refund-policy tightened to stay distinct from its sibling process-refund (the runbook).
GROWN_DESCRIPTION: str = (
    "Decide whether a customer's refund request is allowed under policy — check the order age, the "
    "item's return eligibility, and prior refunds against the refund rules, and return an "
    "allow/deny with the reason. Use to JUDGE a request; not for actually issuing the refund — "
    "process-refund runs that procedure once this skill has allowed it."
)

# --- checks (leave these) ---
assert len(GROWN_DESCRIPTION) > 40, "a description needs enough words to actually discriminate"
print(GROWN_DESCRIPTION)

Decide whether a customer's refund request is allowed under policy — check the order age, the item's return eligibility, and prior refunds against the refund rules, and return an allow/deny with the reason. Use to JUDGE a request; not for actually issuing the refund — process-refund runs that procedure once this skill has allowed it.


## 7. Self-check

There's **no autograding** here — classification and authoring are judgment calls. To check your
work, open [`L2303_lab_solutions.ipynb`](L2303_lab_solutions.ipynb) and compare:

- **Problem 1:** did you match the archetype by *center of gravity* (script vs. rule list vs. ordered
  procedure), not by domain? The two `*-code` / `commit-*` skills are the easy ones to mis-file.
- **Problem 2:** did you name the *wrong* shape (judgment prose) and the *right* one (script-centric),
  and land the fix as "make the body a thin wrapper"?
- **Problem 3:** does your body cover **when / how-to-call / how-to-read** and stay thin — no
  re-deriving the JSON contract in prose?
- **Problem 4 & 5:** are your descriptions **mutually discriminating** — each says what it's for and
  implies what it's *not*, so a model could pick between siblings?

If your wording differs but hits those beats, you're done. Next up: composition — wiring skills like
these into a system ([L2304_lecture.ipynb](L2304_lecture.ipynb)).

[↑ Back to top](#top)
